In [4]:
import pandas as pd

preprocessings = ['MinMax', 'Quantile']
models         = ['kNN', 'LOF', 'IF', 'HBOS', 'AE']
datasets       = ['NSL_KDD', 'UNSW_NB15']
splits         = ['train80', 'abnormal', 'test']

# 변수명: df_{split}_{dataset}_{model}_{preprocessing}
# 예: df_train80_NSL_KDD_kNN_MinMax

for prep in preprocessings:
    for model in models:
        for dataset in datasets:
            for split in splits:
                path = f'{model}_meta_{prep}/{dataset}_{model}_{split}.csv'
                var_name = f'df_{split}_{dataset}_{model}_{prep}'
                globals()[var_name] = pd.read_csv(path)
                print(f'✓ {var_name:50s} ← {path}')

✓ df_train80_NSL_KDD_kNN_MinMax                      ← kNN_meta_MinMax/NSL_KDD_kNN_train80.csv
✓ df_abnormal_NSL_KDD_kNN_MinMax                     ← kNN_meta_MinMax/NSL_KDD_kNN_abnormal.csv
✓ df_test_NSL_KDD_kNN_MinMax                         ← kNN_meta_MinMax/NSL_KDD_kNN_test.csv
✓ df_train80_UNSW_NB15_kNN_MinMax                    ← kNN_meta_MinMax/UNSW_NB15_kNN_train80.csv
✓ df_abnormal_UNSW_NB15_kNN_MinMax                   ← kNN_meta_MinMax/UNSW_NB15_kNN_abnormal.csv
✓ df_test_UNSW_NB15_kNN_MinMax                       ← kNN_meta_MinMax/UNSW_NB15_kNN_test.csv
✓ df_train80_NSL_KDD_LOF_MinMax                      ← LOF_meta_MinMax/NSL_KDD_LOF_train80.csv
✓ df_abnormal_NSL_KDD_LOF_MinMax                     ← LOF_meta_MinMax/NSL_KDD_LOF_abnormal.csv
✓ df_test_NSL_KDD_LOF_MinMax                         ← LOF_meta_MinMax/NSL_KDD_LOF_test.csv
✓ df_train80_UNSW_NB15_LOF_MinMax                    ← LOF_meta_MinMax/UNSW_NB15_LOF_train80.csv
✓ df_abnormal_UNSW_NB15_LOF_MinMax              

In [7]:
# =========================================================
# Meta-Learning (1-Layer NN) - 이상 탐지 앙상블
# 데이터셋: NSL-KDD, UNSW-NB15
# 전처리  : 모든 모델 Quantile 고정
# 입력    : 5개 모델(kNN, LOF, IF, HBOS, AE)의 anomaly score
#
# 사전 준비된 DataFrame (컬럼: anomaly_score, label):
#   df_train80_{dataset}_{model}_Quantile
#   df_abnormal_{dataset}_{model}_Quantile
#   df_test_{dataset}_{model}_Quantile
# =========================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score
)

# =========================================================
# Config
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42
EPOCHS = 20
BATCH  = 128
LR     = 1e-3

MODELS   = ["kNN", "LOF", "IF", "HBOS", "AE"]
DATASETS = ["NSL_KDD", "UNSW_NB15"]
PREP     = "Quantile"

torch.manual_seed(SEED)
np.random.seed(SEED)

# =========================================================
# 1. 준비된 DataFrame을 반복문으로 합쳐 학습/테스트 DataFrame 생성
# =========================================================
def build_dataset(dataset: str) -> tuple:
    parts_train, parts_test = [], []

    for i, model in enumerate(MODELS):
        train80  = eval(f"df_train80_{dataset}_{model}_{PREP}")
        abnormal = eval(f"df_abnormal_{dataset}_{model}_{PREP}")
        test     = eval(f"df_test_{dataset}_{model}_{PREP}")

        train_part = (
            pd.concat([train80, abnormal], ignore_index=True)
            .rename(columns={"anomaly_score": f"score_{model}"})
        )
        test_part = test.rename(columns={"anomaly_score": f"score_{model}"})

        if i == 0:
            parts_train.append(train_part)
            parts_test.append(test_part)
        else:
            parts_train.append(train_part[[f"score_{model}"]])
            parts_test.append(test_part[[f"score_{model}"]])

    df_train = pd.concat(parts_train, axis=1).reset_index(drop=True)
    df_test  = pd.concat(parts_test,  axis=1).reset_index(drop=True)
    return df_train, df_test

# =========================================================
# 2. 전처리: anomaly score → QuantileTransform(normal)
# =========================================================
SCORE_COLS = [f"score_{m}" for m in MODELS]

def preprocess(df_train, df_test):
    qt      = QuantileTransformer(output_distribution="normal", random_state=SEED)
    X_train = qt.fit_transform(df_train[SCORE_COLS].values).astype(np.float32)
    X_test  = qt.transform(df_test[SCORE_COLS].values).astype(np.float32)
    y_train = df_train["label"].values.astype(np.float32)
    y_test  = df_test["label"].values.astype(np.float32)
    return X_train, y_train, X_test, y_test

# =========================================================
# 3. 1-Layer NN 모델
# =========================================================
class OneLayerNN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        criterion(model(X_batch), y_batch).backward()
        optimizer.step()


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for X_batch, y_batch in loader:
        probs.extend(model(X_batch.to(DEVICE)).cpu().numpy())
        labels.extend(y_batch.numpy())
    return np.array(probs), np.array(labels)

# =========================================================
# 4. 파이프라인
# =========================================================
def run_pipeline(dataset: str) -> dict:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    df_train, df_test = build_dataset(dataset)
    X_train, y_train, X_test, y_test = preprocess(df_train, df_test)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
        batch_size=BATCH, shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
        batch_size=BATCH, shuffle=False
    )

    nn_model  = OneLayerNN(input_dim=len(MODELS), hidden_dim=10).to(DEVICE)
    optimizer = torch.optim.Adam(nn_model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.BCELoss()

    for _ in range(EPOCHS):
        train_epoch(nn_model, train_loader, optimizer, criterion)

    probs, y_true = predict(nn_model, test_loader)
    preds_bin     = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, preds_bin, labels=[0, 1]).ravel()

    return {
        "Dataset":     dataset,
        "Recall":      recall_score(y_true, preds_bin, zero_division=0),
        "Precision":   precision_score(y_true, preds_bin, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "F1-score":    f1_score(y_true, preds_bin, zero_division=0),
        "Accuracy":    accuracy_score(y_true, preds_bin),
        "AUC":         roc_auc_score(y_true, probs),
    }

# =========================================================
# 5. 결과 출력
# =========================================================
def print_results(results: list):
    COLS    = ["Recall", "Precision", "Specificity", "F1-score", "Accuracy", "AUC"]
    COLS_KR = ["민감도", "재현율",    "특이도",       "F1 점수",  "정확도",   "AUC"]
    W       = 72

    print("=" * W)
    print(f"  메타러닝 결과  (전처리: {PREP} 고정)")
    print("=" * W)
    print(f"  {'':16s}" + "".join(f"{k:>8}" for k in COLS_KR))
    print("-" * W)

    for r in results:
        label = f"  {r['Dataset']:<16}"
        vals  = "".join(f"{r[c]:>8.2f}" for c in COLS)
        print(label + vals)

    print("-" * W)
    print()

# =========================================================
# 실험 실행
# =========================================================
print(f"Using device: {DEVICE}\n")

results = []
for dataset in DATASETS:
    result = run_pipeline(dataset)
    results.append(result)

print()
print_results(results)

Using device: cpu


  메타러닝 결과  (전처리: Quantile 고정)
                       민감도     재현율     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------
  NSL_KDD             0.89    0.96    0.95    0.93    0.92    0.98
  UNSW_NB15           0.97    0.77    0.65    0.86    0.83    0.95
------------------------------------------------------------------------

